# Notebook 8 - Recommendation Engine

**The capstone notebook.** Everything we've built since Notebook 01 now combines into a deliverable: a per-customer recommendation that marketing can act on.

**Input (from all prior notebooks):**
- `data/processed/customer_segments.parquet` - RFM segments + K-Means clusters (Notebook 04)
- `data/processed/clv_predictions.parquet` - predicted 90-day revenue per-customer (Notebook 05)
- `data/processed/churn_predictions.parquet` - churn risk + strategy quadrant (Notebook 06)
- `data/processed/product_cross_sell.parquet` - products cross-sell rules (Notebook 07)
- `data/interim/transactions_customer_level.parquet` - customer purchase history
- `models/clv_xgboost.joblib`, `models/churn_xgboost.joblib` - trained models with SHAP

**Output:**
- `data/processed/recommendations.parquet` - one row par customer with their complete recommendation
- `data/processed/recommendations.jsonl` - same data as JSON lines, ready for an API or marketing automation system
- `data/customer_recommendation_examples.md` - formatted markdown samples for manager/boss pitch.

## The architecture
For each customer, we produce a rich JSON record:
```
{
    customer_id: 12345,
    segment: 'At Risk',
    strategy_quadrant: 'Urgent Win-back',
    predicted_clv_90d: 312.45,
    churn_risk: 0.68,
    top_churn_drivers: ['recency_day (281d)', 'revenue_last_90d (£0)'],
    recommended_action: 'Personalized win-back email with 15% discount',
    recommended_products: [
        {code: '85099B', desc: 'JUMBO BAG STRAWBERRY', lift: 6.43},
        ...
    ],
    expected_left_value: 47.20,
    priority_score: 0.78
}
```

**Why this matters**: the strategy matrix told us WHO to target. SHAP tells us WHY. Market basket tells us WHAT to offer. This notebook combines all three into a unit that marketing automation can consume directly.

## Notebook structure:
1. Load all inputs
2. Cuild the customer master table (one row per customer with everything joined)
3. Generate SHAP-driven explanations
4. Map each customer to their best product recommendations.
5. Define the action playbook (segment → tactics)
6. Compute priority scores for marketing ranking
7. Build final per-customer recommendation records
8. Save as parquet + JSONL
9. Produce formatted examples for the managers pitch

## Setup

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import json
import joblib
from pathlib import Path
import shap

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:,.2f}'.format)

INTERIM_PATH = Path('../data/interim')
PROCESSED_PATH = Path('../data/processed')
MODELS_PATH = Path('../models')
REPORTS_PATH = Path('../reports/figures')
REPORTS_PATH.mkdir(parents=True, exist_ok=True)

## 1. Load all inputs
Every prior notebook's output is now a building block. Verify everything loads cleanly before building the joins.

In [2]:
segment = pd.read_parquet(PROCESSED_PATH / 'customer_segments.parquet')
clv = pd.read_parquet(PROCESSED_PATH / 'clv_predictions.parquet')
churn = pd.read_parquet(PROCESSED_PATH / 'churn_predictions.parquet')
cross_sell = pd.read_parquet(PROCESSED_PATH / 'product_cross_sell.parquet')
transactions = pd.read_parquet(INTERIM_PATH / 'transactions_customer_level.parquet')

churn_artifact = joblib.load(MODELS_PATH / 'churn_xgboost.joblib')
churn_model = churn_artifact['model']
churn_features = churn_artifact['feature_names']

print(f'Segment:    {len(segment):,} customers')
print(f'CLV preds:  {len(clv):,} customers')
print(f'Churn preds:{len(churn):,} customers')
print(f'Cross-sell: {len(cross_sell):,} product-recommendation pairs ({cross_sell["product_code"].nunique():,} unique source products)')
print(f'Transactions: {len(transactions):,} rows')
print(f'Churn model has {len(churn_features)} features')


Segment:    5,256 customers
CLV preds:  5,256 customers
Churn preds:5,256 customers
Cross-sell: 57 product-recommendation pairs (30 unique source products)
Transactions: 802,637 rows
Churn model has 46 features


## 2. Build the customer master table
Join segments + CLV + churn into one row per customer. This is the foundation everything else attaches to.

In [3]:
master = (
    segment[['CustomerID', 'segment', 'recency_days', 'frequency', 'monetary', 'is_one_time_customer', 'kmeans_cluster', 'cluster_name']]
    .merge(
        clv[['CustomerID', 'predicted_clv_90d', 'clv_decile']],
        on='CustomerID', how='left'
    )
    .merge(
        churn[['CustomerID', 'churn_risk', 'p_active', 'strategy_quadrant']],
        on='CustomerID', how='left'
    )
)

print(f'Master table: {master.shape[0]:,} customers x {master.shape[1]} columns')
master.head(3)

Master table: 5,256 customers x 13 columns


,CustomerID,segment,recency_days,frequency,monetary,is_one_time_customer,kmeans_cluster,cluster_name,predicted_clv_90d,clv_decile,churn_risk,p_active,strategy_quadrant
0,12346.0,Loyal Customers,235,3,"77,352.96",0,3,Top-Tier,9.10,5,0.68,0.32,2. Urgent Win-back
1,12347.0,Champions,39,6,"4,114.18",0,3,Top-Tier,395.21,1,0.12,0.88,1. VIP Retention
2,12348.0,Loyal Customers,158,4,"1,388.40",0,0,Mid-Tier Active,36.74,3,0.31,0.69,1. VIP Retention


## 3. SHAP-driven churn explanations

For each customer, what are the top features pushing them toward churn? This is what makes recommendations explainable rather than black-box.

In [7]:
# Rebuild the feature matrix used ti train the churn model
EXCLUDE_COLS = {
    'CustomerID', 'primary_country', 'RFM_score', 'cluster_name',
    'target_revenue_90d', 'target_orders_90d', 'target_purchased_90d',
}
segment
df_encoded = pd.get_dummies(segment, columns=['segment'], prefix='segment')
feature_cols = [c for c in df_encoded.columns if c not in EXCLUDE_COLS]
X_full = df_encoded[feature_cols].copy()

# Sanity check: features match what the model was trained on
missing_in_X = set(churn_features) - set(X_full.columns)
missing_in_model = set(X_full.columns) - set(churn_features)
if missing_in_X or missing_in_model:
    print(f'⚠️ Feature mismatch! Missing in data: {missing_in_X}, Missing in model: {missing_in_model}')
    X_full = X_full[churn_features] # align to model's feature order
else:
    X_full = X_full[churn_features]
    print('✔︎ Features align with trained model')

✔︎ Features align with trained model


In [8]:
# Compute SHAP values for every customer
explainer = shap.TreeExplainer(churn_model)
shap_values = explainer.shap_values(X_full)
print(f'SHAP values: {shap_values.shape} (customers x features)')

SHAP values: (5256, 46) (customers x features)


In [9]:
def top_churn_driver(customer_idx, shap_arr, X_arr, feature_names, top_n=3):
    """Return top 3 features pushing this customer toward churn.
    Since target=1 mean 'active', the most NEGATIVE SHAP values are the strongest churn drivers.
    """
    contribs = shap_arr[customer_idx]
    feat_values = X_arr.iloc[customer_idx]
    
    # Sort ascending by SHAP value (most negative = most churn-pushing)
    order = np.argsort(contribs[:top_n])
    drivers = []
    for i in order:
        feat_name = feature_names[i]
        feat_value = feat_values.iloc[i]
        shap_value = contribs[i]
        # Coerce any numpy/pandas type to a JSON-friendly Python primitive
        if isinstance(feat_value, (bool, np.bool_)):
            safe_value = bool(feat_value)
        elif isinstance(feat_value, (int, float, np.integer, np.floating)):
            safe_value = float(feat_value)
        else:
            safe_value = float(feat_value)
        # Only report drivers that actually push toward churn (negative)
        if shap_value < 0:
            drivers.append({
                'feature': feat_name,
                'value': safe_value,
                'shap_impact': float(shap_value)
            })
    return drivers

# Test in on customer
test_drivers = top_churn_driver(0, shap_values, X_full, churn_features)
print('Sample top drivers for customer 0:')
for d in test_drivers:
    print(f'    ↓ {d["feature"]} = {d["value"]:.2f} (SHAP {d["shap_impact"]:+.3f})')

Sample top drivers for customer 0:
    ↓ frequency = 3.00 (SHAP -0.024)


## 4. Identify each customer's favorite product

For product recommendations, we need to know what each customer has bought before. We pick their **top products by spend** as the anchor for cross-sell lookup

In [10]:
# For each customer, find their highest-revenue product
customer_top_product = (
    transactions.groupby(['CustomerID','StockCode'], as_index=False)
    .agg(spend=('Revenue', 'sum'), units=('Quantity', 'sum'), desc=('Description', 'first'))
    .sort_values(['CustomerID', 'spend'], ascending=[True, False])
    .groupby('CustomerID')
    .first()
    .reset_index()
    .rename(columns={'StockCode': 'top_product_code', 'desc': 'top_product_decs',
                     'spend': 'top_product_spend', 'units': 'top_product_units'})
)

print(f'Top product computed for {len(customer_top_product):,} customers')
customer_top_product.head(5)

Top product computed for 5,852 customers


,CustomerID,top_product_code,top_product_spend,top_product_units,top_product_decs
0,12346.0,23166,"77,183.60",74215,MEDIUM CERAMIC TOP STORAGE JAR
1,12347.0,84558A,460.20,156,3D DOG PICTURE PLAYING CARDS
2,12348.0,23077,250.00,200,DOUGHNUT LIP GLOSS
3,12349.0,84078A,179.75,5,SET/4 WHITE RETRO STORAGE CUBES
4,12350.0,20615,25.20,12,BLUE POLKADOT PASSPORT COVER


In [14]:
# Build a lookup: stock_code -> list of recommendations
cross_sell_lookup = (
    cross_sell.groupby('product_code')
    .apply(lambda g: g[['recommendation_code', 'recommendation_description','lift']].to_dict(orient='records'))
    .to_dict()
)

# How many of our customers' top product have cross-sell rules?
customers_with_recs = customer_top_product[
    customer_top_product['top_product_code'].isin(cross_sell_lookup.keys())
]
print(f'Customers with at least one product recommendation: {len(customers_with_recs):,} of {len(customer_top_product):,}')
print(f'    ({len(customers_with_recs) / len(customer_top_product) * 100:.1f}% coverage)')


Customers with at least one product recommendation: 675 of 5,852
    (11.5% coverage)


**Note on coverage:** the cross-sell rules from Notebook 07 only cover the ~30 most popular products. <br>
A customer whose top product isn't in that set gets a category-level fallback (most popular product overall) - we'll handle this in the next section.

## 5. Action playbook
Map each strategy quadrant to a concrete recommendation action. This is the rule-based layer on top of the predictions -<br> 
the bridge between "this customer's risk is 0.68" and "do this specific thing."

In [15]:
ACTION_PLAYBOOK = {
    '1. VIP Retention': {
        'action_name': 'Reward & retain',
        'tactic': 'Invite to loyally programl early access to new arrivals',
        'discount_pct': 0,
        'urgency': 'low',
        'expected_intervention_cost_gbp': 5
    },
    '2. Urgent Win-back': {
        'action_name': 'Urgent personalized win-back',
        'tactic': 'Personal email + meaningful discount on a previously-loved category',
        'discount_pct': 15,
        'urgency': 'high',
        'expected_intervention_cost_gbp': 20
    },
    '3. Upsell Opportunity':{
        'action_name': 'Cross-sell to grow basket',
        'tactic': 'Recommend comlementary products at check/email',
        'discount_pct': 5,
        'urgency': 'medium',
        'expected_intervention_cost_gbp': 3
    },
    '4. Low Priority': {
        'action_name': 'Minimal investment',
        'tactic': 'Generic newsletter only',
        'discount_pct': 0,
        'urgency': 'low',
        'expected_intervention_cost_gbp': 0.5
    }
}
for q, p in ACTION_PLAYBOOK.items():
    print(f'{q}: {p["action_name"]} (cost £{p["expected_intervention_cost_gbp"]}, urgency {p["urgency"]})')

1. VIP Retention: Reward & retain (cost £5, urgency low)
2. Urgent Win-back: Urgent personalized win-back (cost £20, urgency high)
3. Upsell Opportunity: Cross-sell to grow basket (cost £3, urgency medium)
4. Low Priority: Minimal investment (cost £0.5, urgency low)
